#### **Step 1 - Chunking** 

**Challenges of Chunking** 

- The fine-tuned model we are using has a hard 1024-token context limit, and `Context + Question + Answer` all has to fit inside that  

- The fine-tuning passages used were typically a few hundred tokens so the model has never seen a 1024-token wall of dumped text

- We need to ensure we split the text in a way that matches what shape of text the model can actually make good use of 

**2 Chunking Strategies**
- **Fixed-size token windows** - Split the document into chunks of exactly `N` tokens, optionally with some overlap between consecutive chunks to prevent accdeitnally cutting a snetence exactly at a chunk boundary 

- **Structure-Aware Splitting** - Split on natural boundaries first (paragraphs, sections, headers), and only fall back to a hard token cut if an individual paragraph is still too long. This produces more semantically coherent chunks - a chunk is more likely to be "one complete idea" rather than "the last half of one idea and the first half of another" 

- For a first implementation, fixed-size windows with overlap are simpler to get right

**Importance of Overlaps**
- Say a document says: “…Faker has won five World Championships. This ties him with T1 as the most decorated organisation in the scene’s history…” 

- If your chunk boundary falls right between those two sentences with zero overlap, one chunk ends with “…five World Championships.” and the next begins with “This ties him…”. 

- A question like “what does Faker winning five championships tie him with?” needs both halves, and a retriever searching chunk-by-chunk
might only surface one of them. 

- A modest overlap (e.g., the last 20% of tokens from one chunk repeated at the start of the next) means a stray boundary is much less likely to sever a fact in half.


**Implementation**

In [ ]:
import tiktoken
enc = tiktoken.get_encoding('gpt2') # we want to encode to tokens so we can guage how large each chunk should be to fit the 1024 context window limit


def chunk_text(text, chunk_size=150, overlap=30):
    """
    Splits a single document's text into overlapping token-based chunks.
    chunk_size: target number of tokens per chunk. Kept well under 1024
    since Question + "Answer:" + generated tokens also need to fit in the same 1024-token window.
    overlap: number of tokens repeated between consecutive chunks,
    so a fact split across a chunk boundary isn't lost.
    """
    token_ids = enc.encode(text) # encodes the text into a list of integer tokens
                                 # e.g. "the cat sat on the math" becomes something like [1169, 3797, 3332, 319, 262, 2603]
                                 # this allows us to chunk by tokens to guarantee we never exceed the context window 
    chunks = [] # store the 150 token long chunks
    start = 0 # starting at the beginning 

    # converts text into tokens and splits this list of tokens into chunks of 150 tokens
    while start < len(token_ids):
        end = start + chunk_size # gets the end of the first chunk 
        chunk_ids = token_ids[start:end] # indexes into the token_ids to get the chunk of tokens
        chunk_text_str = enc.decode(chunk_ids) # decode the chunk into a string
        chunks.append(chunk_text_str) # append that string to chunks so we have a list of chunk strings
        if end >= len(token_ids): # break out once reached the end of exceeded the end
            break
        start = end - overlap # step forward, but re-include the overlap region
    return chunks


# chunk multiple documents making use of chunk_text - here documents are stored as a dict with its id and text content
# wraps each of the chunks with additional metadata such as the document it comes from and its own uique id 
def chunk_documents(documents):
    """
    documents: list of dicts, each like {"id": "...", "text": "..."}
    (one dict per source document/file)
    Returns: list of dicts, each a single chunk:
    {"chunk_id": "...", "doc_id": "...", "text": "..."}
    """
    all_chunks = []
    for doc in documents: # iterate over all the dictionaries in documents - stores 
        pieces = chunk_text(doc["text"]) # obtain the text for that document and chunk it into pieces
        # iterate over every chunk of the document
        # append to a list of dictionaries - each dictionary stores the chunk_id, doc_id and the text chunk
        # this generates a giant list all chunks along with where it came from (doc_id) and it's unique identifier (doc_id)
        for i, piece in enumerate(pieces):
            all_chunks.append({
                "chunk_id": f"{doc['id']}_chunk{i}",
                "doc_id": doc["id"],
                "text": piece,
            })
    return all_chunks

chunks = chunk_documents([{"id": "doc1.txt", "text": "Lee Sang-hyeok, known globally by his in-game moniker Faker, is universally acclaimed as the greatest professional League of Legends player of all time and the definitive icon of esports history. Debuting in 2013 as a teenager, he immediately revolutionized the competitive scene with his unmatched mechanical precision, aggressive mid-lane playstyle, and an uncanny ability to read map movements ahead of his opponents. Over a career spanning more than a decade with a single organization—T1 (formerly SK Telecom T1)—Faker has defied the historically short career spans of esports athletes, maintaining a dominant presence at the highest level of competition across vastly different gameplay eras. His legendary status is cemented by an unparalleled trophy cabinet that features a record-breaking six World Championship titles, multiple Mid-Season Invitational trophies, and numerous domestic LCK championships. Beyond his mechanical virtuosity and deep champion pool, Faker's longevity is a testament to his evolution into a master strategist, an exceptional team leader, and a clutch performer who thrives under immense pressure. As the first inductee into the Hall of Legends, his influence transcends the virtual arena; he has elevated competitive gaming into a mainstream cultural phenomenon, inspired generations of players, and earned a level of global reverence comparable to traditional sporting icons like Michael Jordan or Lionel Messi."}])
chunks

[{'chunk_id': 'doc1.txt_chunk0',
  'doc_id': 'doc1.txt',
  'text': 'Lee Sang-hyeok, known globally by his in-game moniker Faker, is universally acclaimed as the greatest professional League of Legends player of all time and the definitive icon of esports history. Debuting in 2013 as a teenager, he immediately revolutionized the competitive scene with his unmatched mechanical precision, aggressive mid-lane playstyle, and an uncanny ability to read map movements ahead of his opponents. Over a career spanning more than a decade with a single organization—T1 (formerly SK Telecom T1)—Faker has defied the historically short career spans of esports athletes, maintaining a dominant presence at the highest level of competition across vastly different gameplay eras. His legendary status is cemented by an unparalleled trophy cabinet that features a record-breaking'},
 {'chunk_id': 'doc1.txt_chunk1',
  'doc_id': 'doc1.txt',
  'text': " presence at the highest level of competition across vastly dif

#### **Part 1: Chunking**

**What this file does**

Takes raw documents and splits them into small, retrievable pieces that are guaranteed to fit inside GPT-2's 1024-token context window.

**Why chunking is necessary**

The full context window is shared between the retrieved chunk, the question, and the generated answer. Chunking by tokens (not words or characters) is the only way to guarantee the combined prompt never exceeds 1024 tokens.

**Step-by-step**

**1. Tokenise**
- `enc.encode(text)` converts the document string into a flat list of GPT-2 integer token IDs. Slicing is done on this list — not the raw string — so chunks always decode cleanly.

**2. Slice into 150-token windows**
- The while loop walks through the token ID list, taking slices of 150 tokens at a time. Each slice is decoded back into a readable string and stored.

**3. Overlap**
- Instead of advancing 150 tokens each step, the pointer advances only 120 (`end - overlap`). The last 30 tokens of each chunk are repeated at the start of the next. This prevents facts that straddle a chunk boundary from being missed by the retriever.

**4. Tag each chunk with metadata**
- `chunk_documents` wraps `chunk_text` and stores each chunk as a dict with three fields:
    - `chunk_id` — unique identifier used for deduplication in RRF fusion
    - `doc_id` — source file, used to trace answers back to their document
    - `text` — the actual chunk string passed to BM25, FAISS, and ultimately `ask()`

- Metadata is attached alongside the chunk to preserve the strucure of the original documents after shuffling and chunking - the only way to link these chunks back to their original source (document) 

**Pipeline Explained** 

1. Text from documents are chunked and a chunk is stored as a dictionary conisting of its id, doc_id and text. These chunks are all stored in a list (list of dicts)

2. During retrieval **BM25** (keyword search - matches raw text strings) and **FAISS** (semantic search - embeds into vectors + cosine similarity) both search through the chunks independently and each return a ranked list of relevant chunks (relevance by keyword matches and semantic matches)

3. **RRF fusion** combines these two ranked lists into one - in the case the same chunk appears in both lists, `chunk_id` is used to detect identical chunks between the ranked lists and merge their ranks into a single combined score rather than appearing twice

4. The fused ranked list is trimmed to the top-k chunks. Their `text` fields are joined into a single context string and passed into `ask()` alongside the question — this is where the `text` metadata field is consumed, feeding your GPT-2 the retrieved context to generate an answer.

5. `doc_id` is used post-retrieval in evaluation to trace which source file a retrieved chunk came from — allowing you to distinguish between a retrieval failure (wrong document fetched) and a generation failure (right document fetched but GPT-2 still answered incorrectly).

**Output shape**
```python
# input: list of document dicts
[{"id": "faker_bio.txt", "text": "entire document..."}]

# output: flat list of chunk dicts
[
    {"chunk_id": "faker_bio.txt_chunk0", "doc_id": "faker_bio.txt", "text": "first 150 tokens..."},
    {"chunk_id": "faker_bio.txt_chunk1", "doc_id": "faker_bio.txt", "text": "next 150 tokens..."},
]
```

**Document Loader**

In [2]:
import os

# document loader that takes a document and converts it into a dictionary consisting of its id and contents
def load_documents(raw_dir="data/raw"):
    documents = [] # list of dicts for each doc
    for filename in sorted(os.listdir(raw_dir)): # iterate over a list of all filenames in the directory - os.listdir() gives a list of files in the given directory
        if not filename.endswith(".txt"): # if not a txt file then ignore
            continue
        path = os.path.join(raw_dir, filename) # create the path to access the file
        with open(path, "r", encoding="utf-8") as f: # open the txt file
            text = f.read() # read the txt file 
        documents.append({"id": filename, "text": text}) # store that file's contents as a dictionary and append to the list of txt file dicts
    return documents

**Testing**

In [3]:
docs = load_documents("../data")
chunks = chunk_documents(docs)

print(f"{len(docs)} documents -> {len(chunks)} chunks")
print(chunks[0])
print(chunks[1])

1 documents -> 22 chunks
{'chunk_id': 'faker.txt_chunk0', 'doc_id': 'faker.txt', 'text': "Lee Sang-hyeok, known worldwide as Faker, was born on May 7, 1996, in Seoul, South Korea. He rose through the Korean solo queue ladder as a teenager, and his performances there caught the attention of professional scouts. In 2013 he signed with SK Telecom T1 and made his professional debut as the team's mid laner. Within months he had become the centerpiece of the roster and one of the most talked-about newcomers in the history of the game.\n\nFaker's first major triumph came at the 2013 World Championship, where SK Telecom T1 defeated Royal Club in the final held in Los Angeles. He was only seventeen years old. He won his second World Championship in 2015 against KOO Tigers, and"}
{'chunk_id': 'faker.txt_chunk1', 'doc_id': 'faker.txt', 'text': ' Club in the final held in Los Angeles. He was only seventeen years old. He won his second World Championship in 2015 against KOO Tigers, and his third in

#### **Summary of the Process**

1. Raw `.txt` files are loaded from disk and stored as a list of dicts, each with an `id` and `text` field

2. Each document's text is encoded into GPT-2 tokens so chunk sizes are measured accurately against the 1024-token context limit

3. The token list is split into 150-token windows with a 30-token overlap between consecutive chunks to prevent facts being cut at boundaries (`chunk_text` function)

4. Each window is decoded back into a readable string, producing a flat list of chunk strings per document (`chunk_text` function)

5. Each chunk is wrapped in a dict with `chunk_id` (unique identifier), doc_id (source document), and `text` (the chunk string) - (`chunk_documents` function)

6. The final output is a single flat list of all chunk dicts across all documents (`chunk_documents` function) — this is what BM25, FAISS, RRF and generation all operate on